<a href="https://colab.research.google.com/github/mayankchugh-learning/Advance_RAG/blob/main/ParentDocument_RAG/ParentDocument_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parent Document Retriever: Search Small, Answer Big
### Advanced RAG Series — Episode 07
**by Mayank Chugh | IT AI Enthusiast | @itaienthusiast**

---

## Who is this notebook for?

This notebook is a hands-on tutorial for students learning retrieval-augmented generation. You will build a **parent document retriever**: embed and search small child chunks, then hand the LLM the larger parent those chunks came from.

You will build it in this order:

1. **Index** four sample maintenance and cargo documents as child chunks in Chroma and as parents in an in-memory store.
2. **See the baseline fail.** A 300-character search returns the forward-door manual (42 Nm) for an aft-door question whose answer is 38 Nm.
3. **Compare two parent sizes.** Mode 1 returns the whole document. Mode 2 returns a 1,200-character parent chunk.
4. **Guard silent failures** with `id_key="parent_id"`, a relevance threshold of 0.5, and a title prefix on each child.
5. **Generate** with local Qwen3-4B (4-bit), then Claude, then OpenAI, and compare the child-only answer with the parent answer.

> **How to use this notebook:** Run the cells **top to bottom**. Read the markdown above a cell before you run it. The blockquotes named "Saved execution" quote the outputs already stored in this file (run recorded 7 November 2026).

---

## Prerequisites

| Requirement | Details |
|-------------|---------|
| **Runtime** | Google Colab with a **T4 GPU** (Runtime → Change runtime type). The saved run prints `device: cuda`. |
| **Secrets** | `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` are the fallbacks when the local model cannot load. `HF_TOKEN` is optional for this public embedding model. |
| **Data** | No upload. The notebook writes `mro_dataset.py` in the next section. |
| **Concepts** | Embeddings, chunking, and the idea that a retriever returns text for a prompt. |

> **Sample data.** Document numbers, part numbers, torque values, and limits are illustrative. Do not use them for real maintenance or cargo work.

---

## Stack in this episode

| Piece | Choice | Role |
|-------|--------|------|
| Framework | LangChain 1.x, `ParentDocumentRetriever` from `langchain_classic` | Wires child search to parent lookup |
| Embeddings | `BAAI/bge-small-en-v1.5`, 384-dim, normalised | Vectors for the child chunks |
| Child store | Chroma, in memory, one collection per retriever | Similarity search |
| Parent store | `InMemoryStore` | Fetch the parent by id |
| Child splitter | 300 characters, overlap 30 | What gets embedded |
| Parent splitter | 1,200 characters, overlap 0. Omitted in mode 1. | What the LLM reads in mode 2 |
| LLM | `Qwen/Qwen3-4B-Instruct-2507` in NF4 4-bit, then `claude-sonnet-5`, then `gpt-5.6-sol` | `with_fallbacks()` |

---

## Saved execution (source of truth for every number below)

| Check | Printed result |
|-------|----------------|
| Device | `cuda` |
| Documents | 4 documents: 2202, 1543, 1678, and 1052 characters |
| Embedding size | 384 |
| Split | 38 child chunks (avg 172 characters), 7 parent chunks (avg 924 characters) |
| Baseline, k=1 | `SAMPLE-AMM-52-32`, 299 characters, `contains '38 Nm'? False` (the chunk states 42 Nm) |
| Mode 1, whole documents | 4 parents stored, 38 children. The query returns the forward manual (1543 chars, no 38 Nm) then the aft manual (2202 chars, has 38 Nm). Context length 3745. |
| `doc_id` collision | Default `id_key` replaces `doc_id` with `03eb9f45-c87a-489e-8364-2813c9a407e0` |
| Mode 2, `id_key="parent_id"` | Original `doc_id` `SAMPLE-AMM-52-32` survives, plus a `parent_id` UUID. 7 parents, 38 children. |
| Aft question, top children | Distances 0.3703, 0.4327, 0.4659, 0.4774. None of those four children contain `38 Nm`. The fourth parent does (1048 characters). |
| k sweep | k=1 and k=2 omit 38 Nm. k=4 includes it. |
| Relevance | Aft torque 0.738, lithium 0.755, tyre pressure 0.42 (top scores) |
| Threshold 0.5 | 4 parents for the aft question, 0 for the tyre question |
| Titled children | Nearest distance 0.2904 is still the forward manual. The invoke returns three parents; the one that contains 38 Nm is second. |
| Context length | 375 characters (one child) vs 2979 characters (titled parents) |
| LLM load | `loaded: local Qwen3-4B`, `loaded: Claude`, `loaded: OpenAI` |
| Child-chain answer | 42 Nm, cited `SAMPLE-AMM-52-32` |
| Parent-chain answer | 38 Nm (28 lbf-ft), cited `SAMPLE-AMM-52-31` |
| Tyre-pressure answer | `Not found in the provided documents.` |
| Reused collection name | Two adds of 5 chunks → 10 vectors |

---

## Table of contents

| # | Section | What you learn |
|---|---------|----------------|
| 0 | Pipeline | Where children, parents, and the LLM sit |
| 1 | Setup | Packages, secrets, imports |
| 2 | Dataset | Four documents and the four questions, including one with no answer |
| 3 | Embeddings and splitters | 384-dim vectors, 300-char children, 1,200-char parents |
| 4 | Baseline | Why the nearest small chunk is the wrong manual |
| 5 | Mode 1 | Whole-document parents, and the `doc_id` overwrite |
| 6 | Mode 2 | Split parents, `id_key`, the k sweep, near-duplicates |
| 7 | Threshold | A score gate for the question the corpus cannot answer |
| 8 | Titled children | Put the document title into the embedded text |
| 9 | Generation | Same question, child context vs parent context |
| 10 | Collection names | Why a repeated Chroma name appends vectors |


---
# Section 0 — Pipeline

Read this diagram before the code. Later sections fill in one box at a time.

```
┌──────────────────────────────────────────────────────────────────────────┐
│  PHASE 1 — Index                                                         │
│  • Four sample documents (mro_dataset.py)                                │
│  • Child splitter: 300 characters, overlap 30 → embedded in Chroma       │
│  • Parent splitter: 1200 characters, overlap 0 → InMemoryStore           │
│    (Mode 1 skips the parent splitter, so the parent is the whole file)   │
│  • Each child stores the id of its parent                                │
├──────────────────────────────────────────────────────────────────────────┤
│  PHASE 2 — Retrieve                                                      │
│  • Embed the question with BAAI/bge-small-en-v1.5                        │
│  • Chroma returns the nearest k children                                 │
│  • The retriever swaps those children for their parents                  │
│  • Optional gate: similarity_score_threshold 0.5                         │
├──────────────────────────────────────────────────────────────────────────┤
│  PHASE 3 — Generate                                                      │
│  • format_docs stamps each hit with [doc_id | title]                     │
│  • The prompt says: answer only from the context, else the fallback line │
│  • Primary model: Qwen3-4B-Instruct-2507, NF4 4-bit                       │
│  • Fallbacks, used only if the primary raises: Claude, then OpenAI       │
└──────────────────────────────────────────────────────────────────────────┘
```

### Glossary

| Term | Meaning in this notebook |
|------|--------------------------|
| **Child chunk** | Short text that is embedded and searched. 300 characters here. |
| **Parent** | The larger text returned to the LLM. A whole document (mode 1) or a 1,200-character chunk (mode 2). |
| **Docstore** | `InMemoryStore` of parents, keyed by id. Chroma does not hold the parents. |
| **`id_key`** | Metadata field the retriever uses to remember which parent a child belongs to. The default name is `doc_id`, which collides with our own metadata. |
| **Distance** | Number from `similarity_search_with_score`. Smaller is closer. |
| **Relevance score** | Number from `similarity_search_with_relevance_scores`. Larger is a closer match. The 0.5 threshold uses this scale. |
| **Stuff** | The chain concatenates every returned parent into one prompt. |

Distances from the untitled index and distances from the titled index are different vector spaces. Compare rank order inside one index, and compare the printed `has answer` flags across indexes.


---
# Section 1 — Install

The next cell installs the LangChain 1.x stack used in this episode. `ParentDocumentRetriever` is imported from `langchain_classic` because that class moved out of the top-level `langchain` package in 1.x.

| Package | Why this notebook needs it |
|---------|----------------------------|
| `langchain`, `langchain-core`, `langchain-classic` | Documents, prompts, the parent retriever, output parsers |
| `langchain-chroma` | In-memory Chroma vector store |
| `langchain-huggingface` | `HuggingFaceEmbeddings` and the text-generation pipeline wrapper |
| `langchain-text-splitters` | `RecursiveCharacterTextSplitter` |
| `langchain-anthropic`, `langchain-openai` | Fallback chat models |
| `sentence-transformers` | Runs `BAAI/bge-small-en-v1.5` locally |
| `transformers>=4.51.0`, `accelerate`, `bitsandbytes` | Load Qwen3-4B in NF4 4-bit on a T4 |

> **Saved execution:** the install cell finishes with pip's dependency-resolver warnings (`requests`, `oauthlib`, `opentelemetry`) against packages already on the Colab image. Later cells import successfully, so those warnings did not stop this run.


In [1]:
# Install the LangChain 1.x stack. ParentDocumentRetriever is provided by langchain-classic.
# transformers>=4.51.0 is here so Qwen3 loads. bitsandbytes and accelerate are the 4-bit path.
# Saved execution: pip prints dependency conflicts with Colab's preinstalled packages; later imports still run.
!pip install -q langchain langchain-classic langchain-community langchain-core langchain-chroma \
             langchain-huggingface langchain-text-splitters langchain-anthropic langchain-openai \
             sentence-transformers "transformers>=4.51.0" accelerate bitsandbytes


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## Credentials (optional)

Retrieval and embeddings do not need an API key. Generation uses local Qwen when CUDA is available. Claude and OpenAI are attached with `with_fallbacks()` and run only if the model ahead of them raises.

The cell reads Colab Secrets with `userdata.get()` and stores them in `os.environ`. Outside Colab the `ImportError` branch is skipped and the keys stay unset.

| Secret | Used for |
|--------|----------|
| `ANTHROPIC_API_KEY` | `claude-sonnet-5` fallback |
| `OPENAI_API_KEY` | `gpt-5.6-sol` fallback |
| `HF_TOKEN` | Hugging Face downloads, if your account needs a token |

> **Security:** keep keys in Colab Secrets (key icon in the sidebar, notebook access on). Do not paste them into a cell.

> **Saved execution:** all three keys print as `set`.


In [2]:
# Read Colab Secrets into os.environ. Retrieval does not need these keys.
# Generation uses local Qwen on CUDA, then Claude, then OpenAI, and only if the earlier model raises.
# Saved execution: ANTHROPIC_API_KEY, OPENAI_API_KEY, and HF_TOKEN all print as 'set'.
import os
try:
    from google.colab import userdata
    for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except ImportError:
    pass

print({k: ("set" if os.environ.get(k) else "not set") for k in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]})


{'ANTHROPIC_API_KEY': 'set', 'OPENAI_API_KEY': 'set', 'HF_TOKEN': 'set'}


## Imports

One import cell for the rest of the notebook: documents, the parent store, the prompt, Chroma, Hugging Face embeddings, and `ParentDocumentRetriever`.

`device` is `cuda` when the runtime has a GPU and `cpu` otherwise. The embedding model is placed on that device.

> **Saved execution:** `device: cuda`


In [3]:
# Imports shared by every later cell. device follows the runtime: cuda when a GPU is attached.
# Saved execution: device: cuda
import torch
from operator import itemgetter
from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import ParentDocumentRetriever

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


---
# Section 2 — The demo dataset

The next cell writes `mro_dataset.py`. It is sample text written for this episode, with four failures built in so you can see them in the retrieval output.

| doc_id | Title | Saved length | What it is here to show |
|--------|-------|--------------|-------------------------|
| `SAMPLE-AMM-52-31` | Aft cargo door actuator | 2202 chars | The hinge-bolt sentence and the **38 Nm** value sit in different paragraphs, so a short chunk can match the question and still miss the number |
| `SAMPLE-AMM-52-32` | Forward cargo door actuator | 1543 chars | Almost the same wording, with **42 Nm**. This is the near-duplicate that wins small-chunk search |
| `SAMPLE-SOP-DG-07` | Lithium battery acceptance | 1678 chars | Two battery types, with different limits, land in one parent |
| `SAMPLE-SOP-ULD-03` | ULD damage inspection | 1052 chars | A distractor. It is a real document in the index and it is not the answer to these questions |

| Key | Question | Answer in the corpus |
|-----|----------|----------------------|
| `Q1_aft_torque` | Torque for the **aft** cargo door actuator hinge bolts | `38 Nm` |
| `Q2_fwd_torque` | Torque for the **forward** cargo door actuator hinge bolts | `42 Nm` |
| `Q3_lithium_soc` | Maximum state of charge for lithium ion batteries shipped alone | `30 percent` |
| `Q4_zero_answer` | Main landing gear tyre pressure | None. The corpus does not contain it |

> **Sample data, not real records.** Do not use these torque values, part numbers, or limits for real maintenance or cargo work.

> **What to expect:** the cell prints `Writing mro_dataset.py`.


In [4]:
%%writefile mro_dataset.py
"""Sample MRO / air-cargo document set for the Parent Document Retriever episode.

SAMPLE DATA - NOT REAL RECORDS. Document numbers, part numbers, torque values and
limits are illustrative only and must never be used for real maintenance or cargo work.

Seeded edge cases:
  1. Boundary split  - in AMM-52-31 the hinge-bolt sentence and its torque value sit in
                       separate paragraphs, so a small child chunk can match the question
                       but not contain the number.
  2. Near-duplicate  - AMM-52-32 (forward door) mirrors AMM-52-31 (aft door) almost word
                       for word but with different torque values (42 Nm vs 38 Nm).
  3. Zero-answer     - nothing in the corpus covers landing-gear tyre pressure, yet a
                       retriever with no score threshold still returns parents.
  4. Mixed parent    - SOP-DG-07 mixes two battery types (lithium ion vs lithium metal)
                       with different limits inside one parent chunk.
"""

DOCS = [
    {
        "doc_id": "SAMPLE-AMM-52-31",
        "title": "Aft Cargo Door Actuator - Removal and Installation",
        "doc_type": "maintenance_manual",
        "text": """SAMPLE-AMM-52-31 Aft Cargo Door Actuator - Removal and Installation (sample data, not a real manual)

1. General
This task covers removal and installation of the hydraulic actuator that opens and closes the aft cargo door on the SX-320 sample narrowbody. The actuator is attached to the fuselage bracket at its head end and to the door hinge fitting at its rod end. Perform this task when the actuator shows external leakage above the limit in task 52-31-05 or when the door opening time exceeds 18 seconds.

2. Safety Precautions
Make sure the aft cargo door is fully open and the door safety strut is installed before you disconnect the actuator. Release hydraulic pressure from the yellow system and put a DO NOT OPERATE tag on the cargo door control panel. Hydraulic fluid is a skin and eye irritant; wear gloves and goggles.

3. Removal
Disconnect the two hydraulic lines from the actuator and cap them immediately to prevent contamination. Remove the safety wire and the four hinge bolts at the rod end. Support the actuator weight, which is 14 kg, while you remove the head-end attach pin. Discard the hinge bolt washers; they are not reusable.

4. Installation
Put the new actuator in position and install the head-end attach pin with a new cotter pin. Align the rod end with the door hinge fitting.

Install the four hinge bolts (sample P/N SX-5231-HB) through the hinge fitting and actuator rod end. Put one new washer under each bolt head and one under each nut. Make sure the bolt heads face forward so the safety wire can be routed later.

Torque the bolts to 38 Nm (28 lbf-ft) in a cross pattern. After the first 10 door cycles, re-torque to the same value and record the result in the aircraft technical log. Install new safety wire.

5. Test
Connect the hydraulic lines and torque the B-nuts to 25 Nm. Pressurise the yellow system and do three full open and close cycles. Door opening time must be between 12 and 18 seconds. Examine the actuator and line fittings for leaks; no leakage is permitted at the fittings.

6. Close-up
Remove the door safety strut, close the aft cargo door, and remove the DO NOT OPERATE tag. Record the actuator serial number in the component change record.
""",
    },
    {
        "doc_id": "SAMPLE-AMM-52-32",
        "title": "Forward Cargo Door Actuator - Removal and Installation",
        "doc_type": "maintenance_manual",
        "text": """SAMPLE-AMM-52-32 Forward Cargo Door Actuator - Removal and Installation (sample data, not a real manual)

1. General
This task covers removal and installation of the hydraulic actuator that opens and closes the forward cargo door on the SX-320 sample narrowbody. The actuator is attached to the fuselage bracket at its head end and to the door hinge fitting at its rod end. Perform this task when the actuator shows external leakage above the limit in task 52-32-05 or when the door opening time exceeds 16 seconds.

2. Safety Precautions
Make sure the forward cargo door is fully open and the door safety strut is installed before you disconnect the actuator. Release hydraulic pressure from the yellow system and put a DO NOT OPERATE tag on the cargo door control panel.

3. Removal
Disconnect the two hydraulic lines and cap them. Remove the safety wire and the four hinge bolts at the rod end. Support the actuator weight, which is 12 kg, while you remove the head-end attach pin.

4. Installation
Put the new actuator in position and install the head-end attach pin with a new cotter pin. Install the four hinge bolts (sample P/N SX-5232-HB) through the hinge fitting and actuator rod end with new washers. Torque the hinge bolts to 42 Nm (31 lbf-ft) in a cross pattern. Re-torque after the first 10 door cycles. Install new safety wire.

5. Test
Connect the hydraulic lines and torque the B-nuts to 25 Nm. Do three full open and close cycles. Door opening time must be between 10 and 16 seconds. No leakage is permitted at the fittings.
""",
    },
    {
        "doc_id": "SAMPLE-SOP-DG-07",
        "title": "Lithium Battery Shipment Acceptance",
        "doc_type": "cargo_sop",
        "text": """SAMPLE-SOP-DG-07 Lithium Battery Shipment Acceptance (sample data, not a real procedure)

1. Purpose
This procedure tells cargo acceptance staff at the sample hub how to check lithium battery shipments before they are accepted for air transport. It applies to batteries packed alone, packed with equipment, and contained in equipment.

2. Lithium Ion Batteries Packed Alone
Lithium ion cells and batteries shipped by themselves must be offered at a state of charge not exceeding 30 percent of their rated capacity. The shipper must state the state of charge on the declaration. These shipments are cargo aircraft only; reject any booking that routes them on a passenger flight.

3. Lithium Metal Batteries Packed Alone
Lithium metal batteries shipped by themselves are also cargo aircraft only. The state-of-charge rule does not apply to lithium metal batteries; instead check that each cell contains no more than 1 g of lithium metal and each battery no more than 2 g.

4. Packages and Labels
Every package must carry the lithium battery mark and, where required, the Class 9 hazard label. Damaged, crushed or leaking packages must be refused and moved to the quarantine cage at bay D4. Do not open a damaged package on the acceptance floor.

5. Documentation
Check the air waybill handling information, the shipper's declaration, and the cargo-aircraft-only label. Any mismatch between the declaration and the package marks is a rejection reason. Record every rejection in the acceptance log with the reason code DG-REJ.

6. Escalation
If the shipper disputes a rejection, call the duty dangerous goods supervisor. Acceptance staff must not override a rejection on their own.
""",
    },
    {
        "doc_id": "SAMPLE-SOP-ULD-03",
        "title": "ULD Damage Inspection Before Build-Up",
        "doc_type": "cargo_sop",
        "text": """SAMPLE-SOP-ULD-03 ULD Damage Inspection Before Build-Up (sample data, not a real procedure)

1. Purpose
Every unit load device (pallet or container) must be inspected before cargo build-up starts. A damaged ULD can fail in flight or damage the aircraft cargo hold.

2. Pallet Inspection
Check the pallet base for cracks, holes and bending. A pallet with a base deformation of more than 5 cm measured along any edge is unserviceable. Check that the edge rail is continuous and that no seat track fitting is missing.

3. Container Inspection
Check the container walls and door for tears. A tear longer than 10 cm in a fabric door, or any crack in the base, makes the container unserviceable. Check that the door can be closed and latched.

4. Net Inspection
Pallet nets with more than three broken meshes, or any broken corner fitting, must not be used.

5. Unserviceable ULDs
Tag unserviceable units with a red UNSERVICEABLE tag, move them to the ULD repair area, and record the ULD number in the ULD control system. Never build cargo on a tagged unit.
""",
    },
]

QUESTIONS = {
    "Q1_aft_torque": "What torque should the aft cargo door actuator hinge bolts be tightened to?",
    "Q2_fwd_torque": "What torque should the forward cargo door actuator hinge bolts be tightened to?",
    "Q3_lithium_soc": "What is the maximum state of charge for lithium ion batteries shipped alone?",
    "Q4_zero_answer": "What is the main landing gear tyre pressure?",
}

EXPECTED = {
    "Q1_aft_torque": "38 Nm",
    "Q2_fwd_torque": "42 Nm",
    "Q3_lithium_soc": "30 percent",
    "Q4_zero_answer": None,
}

Writing mro_dataset.py


## Load the documents with metadata

Each record becomes a LangChain `Document`. `page_content` is the manual text. `metadata` holds `doc_id`, `title`, and `doc_type`. Later cells print `doc_id` to show which manual came back, so this metadata has to survive indexing.

> **Saved execution:** four lines, lengths 2202, 1543, 1678, and 1052, then `documents: 4`.


In [5]:
# Build one Document per sample record. doc_id is the label later cells print, so it has to stay in metadata.
# Saved execution: lengths 2202, 1543, 1678, 1052, then documents: 4.
from mro_dataset import DOCS, QUESTIONS, EXPECTED

docs = [
    Document(
        page_content=d["text"],
        metadata={"doc_id": d["doc_id"], "title": d["title"], "doc_type": d["doc_type"]},
    )
    for d in DOCS
]

for d in docs:
    print(f"{d.metadata['doc_id']:<18} {len(d.page_content):>5} chars  {d.metadata['title']}")
print("documents:", len(docs))


SAMPLE-AMM-52-31    2202 chars  Aft Cargo Door Actuator - Removal and Installation
SAMPLE-AMM-52-32    1543 chars  Forward Cargo Door Actuator - Removal and Installation
SAMPLE-SOP-DG-07    1678 chars  Lithium Battery Shipment Acceptance
SAMPLE-SOP-ULD-03   1052 chars  ULD Damage Inspection Before Build-Up
documents: 4


---
# Section 3 — Embeddings

`BAAI/bge-small-en-v1.5` is a small English embedding model. This episode uses it so the retrieval cells stay light next to the 4-bit LLM.

`normalize_embeddings=True` L2-normalises every vector. The first call downloads the weights (about 133 MB in the progress bar) and embeds one probe sentence so you can see the width.

> **Saved execution:** `embedding size: 384`


In [6]:
# bge-small-en-v1.5, 384 dimensions, L2-normalised. The probe query forces a download on a cold runtime.
# Saved execution: embedding size: 384
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
print("embedding size:", len(embeddings.embed_query("hinge bolt torque")))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding size: 384


## Two splitters: child and parent

Search wants short text. The LLM wants enough surrounding sentences to see the number and the unit. These two splitters are that split of labour.

| Splitter | `chunk_size` | `chunk_overlap` | Used for |
|----------|--------------|-----------------|----------|
| `child_splitter` | 300 | 30 | Vectors in Chroma |
| `parent_splitter` | 1200 | 0 | Parents in mode 2 |

This cell only previews the counts. It does not build a retriever yet. The same splitter objects are reused later.

> **Saved execution:** `child chunks : 38 | avg chars: 172` and `parent chunks: 7 | avg chars: 924`.


In [7]:
# Preview only. child_splitter is what Chroma embeds. parent_splitter is what mode 2 returns to the LLM.
# Saved execution: 38 children, average 172 characters; 7 parents, average 924 characters.
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=0)

children_preview = child_splitter.split_documents(docs)
parents_preview = parent_splitter.split_documents(docs)
print("child chunks :", len(children_preview), "| avg chars:", round(sum(len(c.page_content) for c in children_preview) / len(children_preview)))
print("parent chunks:", len(parents_preview), "| avg chars:", round(sum(len(p.page_content) for p in parents_preview) / len(parents_preview)))


child chunks : 38 | avg chars: 172
parent chunks: 7 | avg chars: 924


---
# Section 4 — Baseline: small chunks only

Before the parent retriever, search the child chunks directly and keep **one** hit (`k=1`). This is the usual "embed the chunks, return the nearest chunk" pipeline.

The question asks for the **aft** door. The forward manual uses the same hinge-bolt wording and packs `42 Nm` into that same short span. The aft manual puts the hinge-bolt sentence and `38 Nm` in different paragraphs, so the chunk that matches the wording is the forward manual.

> **Saved execution:** `SAMPLE-AMM-52-32 | 299 chars`, the sentence that says `42 Nm`, and `contains '38 Nm'? False`.

The rest of the notebook is about returning a larger parent so the prompt can contain `38 Nm` even when the winning child does not.


In [8]:
# Baseline: nearest child only, k=1. This is the failure the parent retriever is here to change.
# Saved execution: SAMPLE-AMM-52-32, 299 characters, contains '38 Nm'? False. The chunk states 42 Nm.
baseline_store = Chroma.from_documents(
    children_preview, embeddings, collection_name="baseline_small_chunks"
)
baseline_retriever = baseline_store.as_retriever(search_kwargs={"k": 1})

q1 = QUESTIONS["Q1_aft_torque"]
top_child = baseline_retriever.invoke(q1)[0]
print(top_child.metadata["doc_id"], "|", len(top_child.page_content), "chars")
print(top_child.page_content)
print("contains '38 Nm'?", "38 Nm" in top_child.page_content)


SAMPLE-AMM-52-32 | 299 chars
Put the new actuator in position and install the head-end attach pin with a new cotter pin. Install the four hinge bolts (sample P/N SX-5232-HB) through the hinge fitting and actuator rod end with new washers. Torque the hinge bolts to 42 Nm (31 lbf-ft) in a cross pattern. Re-torque after the first
contains '38 Nm'? False


---
# Section 5 — Mode 1: children point at whole documents

`ParentDocumentRetriever` without `parent_splitter` stores each original document as one parent. Children are still the 300-character chunks in Chroma. On a query, the retriever embeds the question, takes `k` children, and replaces them with the unique parents those children point at.

`ids=None` lets the retriever assign parent ids. `search_kwargs={"k": 4}` is how many children are fetched, not how many parents you are guaranteed to receive. Two children from the same document collapse to one parent.

| Argument | Value in this cell | Why |
|----------|--------------------|-----|
| `vectorstore` | Chroma collection `pdr_full_docs_children` | Child vectors |
| `docstore` | new `InMemoryStore` | The four full documents |
| `child_splitter` | 300 / 30 | Same splitter as the preview |
| `parent_splitter` | omitted | Parent = the whole document |
| `search_kwargs` | `k=4` | Fetch four children, then map to parents |

> **Saved execution:** `parents in docstore: 4` and `children in vector store: 38`.


In [9]:
# Mode 1: no parent_splitter, so each parent is the whole document. k=4 is a child count.
# Saved execution: 4 parents in the docstore, 38 children in the vector store.
full_vectorstore = Chroma(collection_name="pdr_full_docs_children", embedding_function=embeddings)
full_docstore = InMemoryStore()

full_doc_retriever = ParentDocumentRetriever(
    vectorstore=full_vectorstore,
    docstore=full_docstore,
    child_splitter=child_splitter,
    search_kwargs={"k": 4},
)
full_doc_retriever.add_documents(docs, ids=None)

print("parents in docstore:", len(list(full_docstore.yield_keys())))
print("children in vector store:", full_vectorstore._collection.count())


parents in docstore: 4
children in vector store: 38


## Query mode 1

Same aft-torque question. Because parents are whole documents, a hit is the entire manual, including sections that have nothing to do with hinge bolts.

The child ranking still prefers the forward manual, so that full document is printed first. The aft manual is second, and it is the one that contains `38 Nm`.

> **Saved execution:**
> - `SAMPLE-AMM-52-32 | 1543 chars | contains '38 Nm': False`
> - `SAMPLE-AMM-52-31 | 2202 chars | contains '38 Nm': True`
> - `context chars sent to the LLM: 3745`

Two lines, not four: the four children belonged to only two documents. 1543 + 2202 = 3745.


In [10]:
# Unique parents of the top children, in child-hit order. Two full manuals, forward manual first.
# Saved execution: 1543 chars without 38 Nm, then 2202 chars with 38 Nm. Context length 3745.
full_hits = full_doc_retriever.invoke(q1)
for h in full_hits:
    print(h.metadata["doc_id"], "|", len(h.page_content), "chars | contains '38 Nm':", "38 Nm" in h.page_content)
print("context chars sent to the LLM:", sum(len(h.page_content) for h in full_hits))


SAMPLE-AMM-52-32 | 1543 chars | contains '38 Nm': False
SAMPLE-AMM-52-31 | 2202 chars | contains '38 Nm': True
context chars sent to the LLM: 3745


## Gotcha: the `doc_id` metadata collision

`ParentDocumentRetriever` records the parent id on each child under `id_key`. The default `id_key` is the string `"doc_id"`. Our documents already use `metadata["doc_id"]` for values like `SAMPLE-AMM-52-31`. The retriever writes its own UUID into that same field, and the original id is gone from the child.

The title is untouched, which is how you can still tell this child is the forward manual.

> **Saved execution:** `doc_id` is `03eb9f45-c87a-489e-8364-2813c9a407e0`, `title` is `Forward Cargo Door Actuator - Removal and Installation`, `doc_type` is `maintenance_manual`.

Mode 2 sets `id_key="parent_id"` so `doc_id` keeps the sample id.


In [11]:
# Default id_key is doc_id, so the retriever's parent UUID overwrites our sample id on the child.
# Saved execution: doc_id 03eb9f45-c87a-489e-8364-2813c9a407e0, title still the forward cargo door manual.
child = full_vectorstore.similarity_search(q1, k=1)[0]
print(child.metadata)


{'title': 'Forward Cargo Door Actuator - Removal and Installation', 'doc_id': '03eb9f45-c87a-489e-8364-2813c9a407e0', 'doc_type': 'maintenance_manual'}


---
# Section 6 — Mode 2: children point at parent chunks

Whole documents fix the missing sentence and also drag in every other section. This mode sets `parent_splitter` (1,200 characters, overlap 0) so the LLM receives the parent **chunk** that contains the matched child.

`id_key="parent_id"` stores the link in a new metadata field. `doc_id` stays `SAMPLE-AMM-52-32` (or whichever manual the child came from).

This retriever, named `retriever`, is the one the inspect cells and the k sweep use. `k` starts at 4.

> **Saved execution:** the nearest child still has `doc_id` `SAMPLE-AMM-52-32` and a `parent_id` UUID. Then `parent chunks in docstore: 7` and `child chunks in vector store: 38` (the same 38 children as the preview).


In [12]:
# Mode 2: 1200-character parents, and id_key='parent_id' so metadata['doc_id'] keeps the sample id.
# Saved execution: doc_id SAMPLE-AMM-52-32 plus a parent_id UUID. 7 parents, 38 children.
chunk_vectorstore = Chroma(collection_name="pdr_split_children", embedding_function=embeddings)
chunk_docstore = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=chunk_vectorstore,
    docstore=chunk_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    id_key="parent_id",
    search_kwargs={"k": 4},
)
retriever.add_documents(docs)
print(chunk_vectorstore.similarity_search(q1, k=1)[0].metadata)

print("parent chunks in docstore:", len(list(chunk_docstore.yield_keys())))
print("child chunks in vector store:", chunk_vectorstore._collection.count())


{'doc_type': 'maintenance_manual', 'parent_id': '2d88ba57-1f18-4136-9071-9a56111a996a', 'doc_id': 'SAMPLE-AMM-52-32', 'title': 'Forward Cargo Door Actuator - Removal and Installation'}
parent chunks in docstore: 7
child chunks in vector store: 38


## Inspect: children vs parents for the aft question

`inspect` prints two views of one question:

- **Children** from `similarity_search_with_score`. The number is a distance (smaller is closer). `has answer` is true only when the expected string is inside that child's text.
- **Parents** from `retriever.invoke`. Same `has answer` test on the parent text.

For Q1 the expected string is `38 Nm`. Watch for a parent that contains it when the child that led there does not. That gap is the reason to return parents.

> **Saved execution for Q1:**
> - Children, all `has answer` False: `SAMPLE-AMM-52-32` 0.3703, `SAMPLE-AMM-52-31` 0.4327, `SAMPLE-AMM-52-32` 0.4659, `SAMPLE-AMM-52-31` 0.4774
> - Four parents. The first three do not contain `38 Nm`. The last does: `SAMPLE-AMM-52-31`, 1048 characters.


In [13]:
# Distance is smaller-is-closer. has answer is True only when the expected string is inside that text.
# Saved Q1: all four children lack '38 Nm'. The parent that contains it is last: SAMPLE-AMM-52-31, 1048 chars.
def inspect(question, expected):
    children = chunk_vectorstore.similarity_search_with_score(question, k=4)
    parents = retriever.invoke(question)
    print("Q:", question)
    print("  top children (doc_id | distance | has answer):")
    for c, score in children:
        print(f"    {c.metadata['doc_id']:<18} {score:.4f}  {bool(expected) and expected in c.page_content}")
    print("  parents returned:", len(parents))
    for p in parents:
        print(f"    {p.metadata['doc_id']:<18} {len(p.page_content):>5} chars  has answer: {bool(expected) and expected in p.page_content}")
    return children, parents

q1_children, q1_parents = inspect(q1, EXPECTED["Q1_aft_torque"])


Q: What torque should the aft cargo door actuator hinge bolts be tightened to?
  top children (doc_id | distance | has answer):
    SAMPLE-AMM-52-32   0.3703  False
    SAMPLE-AMM-52-31   0.4327  False
    SAMPLE-AMM-52-32   0.4659  False
    SAMPLE-AMM-52-31   0.4774  False
  parents returned: 4
    SAMPLE-AMM-52-32     556 chars  has answer: False
    SAMPLE-AMM-52-31    1151 chars  has answer: False
    SAMPLE-AMM-52-32     984 chars  has answer: False
    SAMPLE-AMM-52-31    1048 chars  has answer: True


## How many children should we fetch?

`k` is the number of **children** fetched before the parent lookup. The parent that holds `38 Nm` is behind three closer children, so a small `k` never requests it.

The loop prints k=1, k=2, and k=4, then sets `search_kwargs` back to `k=4` so the following cells keep that setting.

> **Saved execution:**
> - `k=1`: `['SAMPLE-AMM-52-32']`, 38 Nm absent
> - `k=2`: forward manual, then an aft parent, 38 Nm absent (that aft parent is the 1151-character chunk from the inspect cell, which does not contain the torque sentence)
> - `k=4`: four parents, 38 Nm present


In [14]:
# k is how many children are fetched. The 38 Nm parent is the fourth hit, so k=1 and k=2 miss it.
# The loop restores search_kwargs to k=4 for the cells below.
# Saved execution: 38 Nm present is False, False, then True.
for k in [1, 2, 4]:
    retriever.search_kwargs = {"k": k}
    found = retriever.invoke(q1)
    print(f"k={k}: parents={[p.metadata['doc_id'] for p in found]} | 38 Nm present: {any('38 Nm' in p.page_content for p in found)}")
retriever.search_kwargs = {"k": 4}


k=1: parents=['SAMPLE-AMM-52-32'] | 38 Nm present: False
k=2: parents=['SAMPLE-AMM-52-32', 'SAMPLE-AMM-52-31'] | 38 Nm present: False
k=4: parents=['SAMPLE-AMM-52-32', 'SAMPLE-AMM-52-31', 'SAMPLE-AMM-52-32', 'SAMPLE-AMM-52-31'] | 38 Nm present: True


## Near-duplicate check: the forward door question

Q2 asks for the forward door, and `42 Nm` sits inside the nearest child. Parent lookup is unnecessary for this wording. The cell is here so you can see the same index behave when the number and the matching sentence share a chunk.

> **Saved execution:** top child `SAMPLE-AMM-52-32` at distance 0.3778, `has answer` True. First parent is that same manual, 556 characters, `has answer` True. The other three parents are the near-duplicate aft chunks and the other forward chunk.


In [15]:
# Control question. 42 Nm sits inside the nearest child, so the first parent already has the answer.
# Saved execution: SAMPLE-AMM-52-32 at distance 0.3778, has answer True.
q2_children, q2_parents = inspect(QUESTIONS["Q2_fwd_torque"], EXPECTED["Q2_fwd_torque"])


Q: What torque should the forward cargo door actuator hinge bolts be tightened to?
  top children (doc_id | distance | has answer):
    SAMPLE-AMM-52-32   0.3778  True
    SAMPLE-AMM-52-31   0.4454  False
    SAMPLE-AMM-52-31   0.4618  False
    SAMPLE-AMM-52-32   0.4862  False
  parents returned: 4
    SAMPLE-AMM-52-32     556 chars  has answer: True
    SAMPLE-AMM-52-31    1151 chars  has answer: False
    SAMPLE-AMM-52-31    1048 chars  has answer: False
    SAMPLE-AMM-52-32     984 chars  has answer: False


## Lithium batteries: many children, one parent

Q3's four nearest children all come from `SAMPLE-SOP-DG-07`, and they map to a single parent. That parent contains the 30 percent rule and the start of the lithium-metal section (the different limit the dataset was written to mix into one chunk).

> **Saved execution:** top child distance 0.3467 with `has answer` True. `parents returned: 1`, 969 characters, `has answer` True. The preview print is the first 900 characters of that parent, and it includes "not exceeding 30 percent".


In [16]:
# Four children from one SOP collapse to one parent. That parent also starts the lithium-metal limits.
# Saved execution: 1 parent, 969 characters, has answer True. Top child distance 0.3467.
q3_children, q3_parents = inspect(QUESTIONS["Q3_lithium_soc"], EXPECTED["Q3_lithium_soc"])
print()
print(q3_parents[0].page_content[:900])


Q: What is the maximum state of charge for lithium ion batteries shipped alone?
  top children (doc_id | distance | has answer):
    SAMPLE-SOP-DG-07   0.3467  True
    SAMPLE-SOP-DG-07   0.3950  False
    SAMPLE-SOP-DG-07   0.4605  False
    SAMPLE-SOP-DG-07   0.5444  False
  parents returned: 1
    SAMPLE-SOP-DG-07     969 chars  has answer: True

SAMPLE-SOP-DG-07 Lithium Battery Shipment Acceptance (sample data, not a real procedure)

1. Purpose
This procedure tells cargo acceptance staff at the sample hub how to check lithium battery shipments before they are accepted for air transport. It applies to batteries packed alone, packed with equipment, and contained in equipment.

2. Lithium Ion Batteries Packed Alone
Lithium ion cells and batteries shipped by themselves must be offered at a state of charge not exceeding 30 percent of their rated capacity. The shipper must state the state of charge on the declaration. These shipments are cargo aircraft only; reject any booking that route

---
# Section 7 — The question with no answer

Nothing in the four documents states a main-landing-gear tyre pressure. A retriever with no score gate still returns the nearest parents, and those parents read like confident context.

`inspect` uses distance. The loop under it calls `similarity_search_with_relevance_scores`, which is a different number: larger means a closer match. The threshold in the next cell is on that relevance scale.

> **Saved execution:**
> - Four children, all `SAMPLE-AMM-52-31`, distances 0.8201, 0.8317, 0.8320, 0.8323, `has answer` False
> - 2 parents returned, 1048 and 1151 characters, neither has an answer
> - Top relevance: Q1 **0.738**, Q3 **0.755**, Q4 **0.42** (full lists: Q1 0.738, 0.694, 0.671, 0.662; Q3 0.755, 0.721, 0.674, 0.615; Q4 0.42, 0.412, 0.412, 0.411)


In [17]:
# The corpus has no tyre pressure. The retriever still returns parents. Relevance (larger is closer) shows the gap.
# Saved relevance, top scores: Q1 0.738, Q3 0.755, Q4 0.42. Q4 returns 2 parents, neither has an answer.
q4_children, q4_parents = inspect(QUESTIONS["Q4_zero_answer"], EXPECTED["Q4_zero_answer"])
for name in ["Q1_aft_torque", "Q3_lithium_soc", "Q4_zero_answer"]:
    scores = [round(s, 3) for _, s in chunk_vectorstore.similarity_search_with_relevance_scores(QUESTIONS[name], k=4)]
    print(f"{name:<15} relevance: {scores}")


Q: What is the main landing gear tyre pressure?
  top children (doc_id | distance | has answer):
    SAMPLE-AMM-52-31   0.8201  False
    SAMPLE-AMM-52-31   0.8317  False
    SAMPLE-AMM-52-31   0.8320  False
    SAMPLE-AMM-52-31   0.8323  False
  parents returned: 2
    SAMPLE-AMM-52-31    1048 chars  has answer: False
    SAMPLE-AMM-52-31    1151 chars  has answer: False
Q1_aft_torque   relevance: [0.738, 0.694, 0.671, 0.662]
Q3_lithium_soc  relevance: [0.755, 0.721, 0.674, 0.615]
Q4_zero_answer  relevance: [0.42, 0.412, 0.412, 0.411]


## Guard it with a relevance threshold

`search_type="similarity_score_threshold"` drops children whose relevance score is below `score_threshold` before any parent is fetched. This cell reuses the mode 2 vector store and docstore. It does not re-embed.

The threshold 0.5 sits between the aft question (top relevance 0.738, and all four of its scores are above 0.5) and the tyre question (top relevance 0.42).

> **Saved execution:** a LangChain warning, `No relevant docs were retrieved using the relevance score threshold 0.5`, then `Q1_aft_torque parents returned: 4` and `Q4_zero_answer parents returned: 0`.


In [18]:
# Drop children under relevance 0.5 before the parent lookup. Same mode-2 store, no re-embedding.
# Saved execution: warning for the threshold, then 4 parents on Q1 and 0 parents on Q4.
guarded_retriever = ParentDocumentRetriever(
    vectorstore=chunk_vectorstore,
    docstore=chunk_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    id_key="parent_id",
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)
for name in ["Q1_aft_torque", "Q4_zero_answer"]:
    print(f"{name:<15} parents returned: {len(guarded_retriever.invoke(QUESTIONS[name]))}")


Q1_aft_torque   parents returned: 4
Q4_zero_answer  parents returned: 0


---
# Section 8 — Title the children

The forward and aft manuals share most of their sentences. The titles do not. `TitledChildSplitter` prefixes every child with `metadata["title"]` **before** the child is embedded. The parents stored in the docstore stay the original text, so the LLM still reads the manual rather than a title-prefixed copy.

This retriever also keeps the 0.5 relevance threshold. Distances here are from the titled index. They are not comparable to the 0.37 distances in section 6.

> **Saved execution (distance, smaller is closer):**
> - `SAMPLE-AMM-52-32` **0.2904** — forward manual, still the nearest child
> - `SAMPLE-AMM-52-31` 0.3204, 0.3231, 0.3381 — three aft children. The 0.3231 snippet starts `Torque the bolts to 38`
> - Parents from `invoke`: `('SAMPLE-AMM-52-32', False)`, `('SAMPLE-AMM-52-31', True)`, `('SAMPLE-AMM-52-31', False)`

The parent that contains 38 Nm is second. The threshold leaves three parents rather than four.


In [19]:
# Prefix each child with its document title before embedding. Parents in the docstore stay original text.
# Saved distances: forward manual 0.2904, then aft children 0.3204, 0.3231, 0.3381.
# Saved parents: forward (no 38 Nm), aft parent that contains 38 Nm, another aft parent that does not.
class TitledChildSplitter(RecursiveCharacterTextSplitter):
    def split_documents(self, documents):
        chunks = super().split_documents(documents)
        for c in chunks:
            c.page_content = f"{c.metadata['title']}: {c.page_content}"
        return chunks

titled_vectorstore = Chroma(collection_name="pdr_titled_children", embedding_function=embeddings)
titled_docstore = InMemoryStore()
titled_retriever = ParentDocumentRetriever(
    vectorstore=titled_vectorstore,
    docstore=titled_docstore,
    child_splitter=TitledChildSplitter(chunk_size=300, chunk_overlap=30),
    parent_splitter=parent_splitter,
    id_key="parent_id",
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.5},
)
titled_retriever.add_documents(docs)

for c, score in titled_vectorstore.similarity_search_with_score(q1, k=4):
    print(f"{c.metadata['doc_id']:<18} {score:.4f}  {c.page_content[:75]!r}")
print([(p.metadata["doc_id"], "38 Nm" in p.page_content) for p in titled_retriever.invoke(q1)])


SAMPLE-AMM-52-32   0.2904  'Forward Cargo Door Actuator - Removal and Installation: Put the new actuato'
SAMPLE-AMM-52-31   0.3204  'Aft Cargo Door Actuator - Removal and Installation: Install the four hinge '
SAMPLE-AMM-52-31   0.3231  'Aft Cargo Door Actuator - Removal and Installation: Torque the bolts to 38 '
SAMPLE-AMM-52-31   0.3381  'Aft Cargo Door Actuator - Removal and Installation: Disconnect the two hydr'
[('SAMPLE-AMM-52-32', False), ('SAMPLE-AMM-52-31', True), ('SAMPLE-AMM-52-31', False)]


---
# Section 9 — Generation

### Load order

`load_llm` builds a list and returns the first model wrapped with `with_fallbacks` for the rest. A fallback runs when the model ahead of it raises. A successful answer from the primary model stays on that model.

| Order | When it is added | Model | Settings in this cell |
|-------|------------------|-------|------------------------|
| 1 | CUDA is available | `Qwen/Qwen3-4B-Instruct-2507` via `ChatHuggingFace` | NF4, `bnb_4bit_compute_dtype=torch.float16`, `max_new_tokens=512`, `do_sample=False`, `return_full_text=False` |
| 2 | `ANTHROPIC_API_KEY` is set | `claude-sonnet-5` | `temperature=0` |
| 3 | `OPENAI_API_KEY` is set | `gpt-5.6-sol` | `temperature=0` |

If the runtime has no GPU and neither key is set, the cell raises `RuntimeError` and tells you to switch to a T4 or add a secret.

> **Saved execution:** the Qwen weight files download, then `loaded: local Qwen3-4B`, `loaded: Claude`, `loaded: OpenAI`.


In [20]:
# Primary model is local Qwen3-4B in NF4 when CUDA is available. Claude and OpenAI are fallbacks.
# do_sample=False and max_new_tokens=512. with_fallbacks runs the next model only if the earlier one raises.
# Saved execution: loaded local Qwen3-4B, Claude, and OpenAI.
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

def load_local_llm():
    from transformers import BitsAndBytesConfig
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    pipe = HuggingFacePipeline.from_model_id(
        model_id="Qwen/Qwen3-4B-Instruct-2507",
        task="text-generation",
        device_map="auto",
        model_kwargs={"quantization_config": quant},
        pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "return_full_text": False},
    )
    return ChatHuggingFace(llm=pipe)

def load_llm():
    candidates = []
    if torch.cuda.is_available():
        candidates.append(("local Qwen3-4B", load_local_llm))
    if os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        candidates.append(("Claude", lambda: ChatAnthropic(model="claude-sonnet-5", temperature=0)))
    if os.environ.get("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        candidates.append(("OpenAI", lambda: ChatOpenAI(model="gpt-5.6-sol", temperature=0)))
    if not candidates:
        raise RuntimeError("No GPU and no API key: switch to a T4 runtime or add a key in Colab Secrets.")
    models = []
    for name, factory in candidates:
        try:
            models.append(factory())
            print("loaded:", name)
        except Exception as e:
            print("skipped:", name, "->", type(e).__name__)
    return models[0].with_fallbacks(models[1:]) if len(models) > 1 else models[0]

llm = load_llm()


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loaded: local Qwen3-4B
loaded: Claude
loaded: OpenAI


## Two chains, one prompt

Both chains share `rag_prompt`:

- Answer only from the context.
- Quote the value and cite the `doc_id`.
- If the context does not contain the answer, reply exactly `Not found in the provided documents.`

`format_docs` puts `[doc_id | title]` above each document so the citation has a source line to copy.

| Chain | Retriever | What the prompt receives on Q1 |
|-------|-----------|--------------------------------|
| `child_chain` | Baseline Chroma, `k=1` | The single forward-door chunk |
| `parent_chain` | `titled_retriever` from section 8 (split parents, titles on the children, threshold 0.5) | The parents of the titled search |

Mode 1's whole documents are not what these chains send. The parent chain is the titled, split-parent retriever.

> **Saved execution:** `child-only k=1  context chars: 375` and `parent (titled) context chars: 2979`.


In [21]:
# child_chain stuffs the single baseline chunk. parent_chain stuffs the titled parents from section 8.
# Saved Q1 context length: 375 characters (child) and 2979 characters (titled parents).
def format_docs(documents):
    return "\n\n".join(
        f"[{d.metadata['doc_id']} | {d.metadata['title']}]\n{d.page_content}" for d in documents
    )

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a maintenance and cargo documentation assistant. Answer only from the context. "
     "Quote the value and cite the doc_id. If the context does not contain the answer, reply exactly: "
     "Not found in the provided documents."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

parent_chain = (
    {"context": itemgetter("question") | titled_retriever | format_docs, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)
child_chain = (
    {"context": itemgetter("question") | baseline_store.as_retriever(search_kwargs={"k": 1}) | format_docs,
     "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

for name, chain_retriever in [("child-only k=1", baseline_store.as_retriever(search_kwargs={"k": 1})), ("parent (titled)", titled_retriever)]:
    ctx = format_docs(chain_retriever.invoke(q1))
    print(f"{name:<15} context chars: {len(ctx)}")


child-only k=1  context chars: 375
parent (titled) context chars: 2979


## Answer with the child-only chain

The prompt for this call contains the 299-character forward-door chunk (375 characters after `format_docs` adds the header). That chunk states 42 Nm. The aft manual is not in the prompt.

> **Saved execution:** the printed answer says the bolts should be tightened to **42 Nm (31 lbf-ft)** and cites `[SAMPLE-AMM-52-32 | Forward Cargo Door Actuator - Removal and Installation]`.


In [22]:
# The child prompt contains the forward-door chunk, which states 42 Nm.
# Saved answer: 42 Nm (31 lbf-ft), cited SAMPLE-AMM-52-32.
print(child_chain.invoke({"question": q1}))


The aft cargo door actuator hinge bolts should be tightened to 42 Nm (31 lbf-ft).  

Source: [SAMPLE-AMM-52-32 | Forward Cargo Door Actuator - Removal and Installation]


## Answer with the parent chain

The prompt now contains the titled parents (2979 characters). One of them is the aft installation chunk that states 38 Nm.

> **Saved execution:** the printed answer says **38 Nm (28 lbf-ft)**, quotes `Torque the bolts to 38 Nm (28 lbf-ft) in a cross pattern.`, and cites `SAMPLE-AMM-52-31`.


In [23]:
# The parent prompt contains the aft installation chunk, which states 38 Nm.
# Saved answer: 38 Nm (28 lbf-ft), quoted from the manual, cited SAMPLE-AMM-52-31.
print(parent_chain.invoke({"question": q1}))


The aft cargo door actuator hinge bolts should be tightened to **38 Nm (28 lbf-ft)**.  

This is stated in the context under "Installation" for [SAMPLE-AMM-52-31 | Aft Cargo Door Actuator - Removal and Installation]:  
"Torque the bolts to 38 Nm (28 lbf-ft) in a cross pattern."  

Doc_id: SAMPLE-AMM-52-31


## Answer the unanswerable question

Q4 still goes through `parent_chain`, so the 0.5 threshold applies. The tyre-pressure children scored about 0.42, which is below the gate, so the retriever returns no documents. The prompt's context is empty, and the instruction says to use the fallback sentence.

> **Saved execution:** the same threshold warning as section 7, then `Not found in the provided documents.`


In [24]:
# Q4 is under the 0.5 threshold, so the prompt context is empty and the fallback sentence applies.
# Saved answer: Not found in the provided documents.
print(parent_chain.invoke({"question": QUESTIONS["Q4_zero_answer"]}))


Not found in the provided documents.


---
# Section 10 — Reusing a Chroma collection name

`Chroma(collection_name=...)` opens the collection if that name already exists in this process. A second object with the same name is the same collection. `add_documents` appends.

This cell adds the first five child chunks, then adds those same five chunks again under `shared_name_demo`.

> **Saved execution:** `children added twice under one collection name -> 10 vectors`.

Use a distinct `collection_name` for each retriever in one runtime (`baseline_small_chunks`, `pdr_full_docs_children`, `pdr_split_children`, `pdr_titled_children`). Running this demonstration cell again in the same runtime appends another batch to `shared_name_demo`.


In [25]:
# A second Chroma object with the same collection_name is the same collection. add_documents appends.
# Saved execution: five chunks added twice -> 10 vectors.
store_a = Chroma(collection_name="shared_name_demo", embedding_function=embeddings)
store_a.add_documents(children_preview[:5])
store_b = Chroma(collection_name="shared_name_demo", embedding_function=embeddings)
store_b.add_documents(children_preview[:5])
print("children added twice under one collection name ->", store_b._collection.count(), "vectors")


children added twice under one collection name -> 10 vectors


---
# Key lessons (checked against the saved outputs)

Work through these against the printed cells. Each one names the number to look for.

1. **Small-chunk search returned the other manual.** Baseline `k=1` printed `SAMPLE-AMM-52-32`, 299 characters, `contains '38 Nm'? False`, and the chunk text states 42 Nm.
2. **The parent can hold the number when the child does not.** On Q1 every top child has `has answer` False (distances 0.3703, 0.4327, 0.4659, 0.4774). The fourth parent is `SAMPLE-AMM-52-31`, 1048 characters, `has answer` True.
3. **`k` counts children.** k=1 and k=2 omit 38 Nm. k=4 includes it. At k=2 the aft parent that does come back is the chunk without the torque sentence.
4. **Name the link field yourself.** Default `id_key` wrote `03eb9f45-c87a-489e-8364-2813c9a407e0` into `doc_id`. `id_key="parent_id"` left `doc_id` as `SAMPLE-AMM-52-32`.
5. **A threshold needs the relevance scale.** Q4's best relevance is 0.42 and its best distance is 0.8201. Threshold 0.5 returned 0 parents for Q4 and 4 parents for Q1 (top relevance 0.738).
6. **A title prefix moves rank, and the forward manual can still be first.** Titled distances: 0.2904 (forward), then 0.3204, 0.3231, 0.3381. The parent with 38 Nm was second of the three parents returned.
7. **The two answer cells follow the context they were given.** Child chain: 375 characters, answer 42 Nm, cite `SAMPLE-AMM-52-32`. Parent chain: 2979 characters, answer 38 Nm (28 lbf-ft), cite `SAMPLE-AMM-52-31`. Empty context on the tyre question: `Not found in the provided documents.`
8. **A repeated Chroma collection name appends.** 5 chunks added twice produced 10 vectors.

### Self-check

- Which paragraph boundary in `SAMPLE-AMM-52-31` separates the hinge-bolt sentence from 38 Nm?
- Why did k=2 return an aft parent and still leave 38 Nm out?
- Why is 0.42 a drop under a 0.5 gate, while a distance of 0.29 is a close neighbour?

The next folder in the series index is contextual compression: [`../ContextualCompression_RAG/`](../ContextualCompression_RAG/).


## Pipeline summary — saved execution (7 November 2026)

| Component | Class / API | Model / value | Role |
|-----------|-------------|---------------|------|
| Corpus | `mro_dataset.py` | 4 sample documents | Aft manual, forward near-duplicate, lithium SOP, ULD SOP |
| Embeddings | `HuggingFaceEmbeddings` | `BAAI/bge-small-en-v1.5`, 384-dim, normalised | Child vectors. Saved device: `cuda` |
| Child splitter | `RecursiveCharacterTextSplitter` | 300 / overlap 30 | 38 chunks, average 172 characters |
| Parent splitter | `RecursiveCharacterTextSplitter` | 1200 / overlap 0 | 7 chunks, average 924 characters. Omitted in mode 1 |
| Child store | `Chroma` | One in-memory collection per retriever | Similarity search |
| Parent store | `InMemoryStore` | — | Parent text fetched by id |
| Retriever | `ParentDocumentRetriever` | `k=4`, and later `id_key="parent_id"` | Children in, parents out |
| Score gate | `similarity_score_threshold` | `score_threshold=0.5` | Q1 returns 4 parents; Q4 returns 0 |
| Title trick | `TitledChildSplitter` | Title prefixed on children only | Nearest child still the forward manual at distance 0.2904 |
| Prompt | `ChatPromptTemplate` | Answer only from context; else the fallback sentence | Shared by both chains |
| LLM | `ChatHuggingFace` + `with_fallbacks` | Qwen3-4B NF4, then `claude-sonnet-5`, then `gpt-5.6-sol` | Saved load line lists all three. Answer cells completed on this run |
| Child-chain result | baseline `k=1` | 375 context characters | Printed answer: 42 Nm, `SAMPLE-AMM-52-32` |
| Parent-chain result | titled retriever | 2979 context characters | Printed answer: 38 Nm (28 lbf-ft), `SAMPLE-AMM-52-31` |
| Unanswerable result | same parent chain | empty context under the 0.5 gate | Printed answer: `Not found in the provided documents.` |

---
*Mayank Chugh | GitHub: [mayankchugh-learning](https://github.com/mayankchugh-learning) | YouTube: [@itaienthusiast](https://www.youtube.com/@itaienthusiast)*
